# Generate SatCLIP Embeddings
## Specifically for S2-100K

In [1]:
import os
import sys
from pathlib import Path
from tqdm import tqdm

In [2]:
sys.path.append("..")

In [3]:
import numpy as np
import torch
from torch.utils.data import DataLoader
import pandas as pd
from huggingface_hub import hf_hub_download
from satclip.satclip.load import get_satclip

from modeling import LocationEmbeddingModel

/media/volume/xAi-data/miniconda3/envs/xai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

/media/volume/xAi-data/miniconda3/envs/xai/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


In [5]:
location_model = "microsoft/SatCLIP-ViT16-L10"
location_model_filename = "satclip-vit16-l10.ckpt"

In [6]:
model = get_satclip(hf_hub_download(location_model, location_model_filename), device="cpu")

using pretrained moco vit16


In [7]:
parent = Path.cwd().parent

In [8]:
satclip_df = pd.read_csv(str(parent / "text-descriptions/index_with_ee.csv"))

In [9]:
satclip_df

,fn,lat,lon,avg_evi,avg_ndvi,biome,country,ecoregion,elevation_m,nightlights,...,realm,soil_carbon_g_kg,soil_ph,state,temp_annual_C,temp_max_warmest_C,temp_min_coldest_C,temp_range_C,tree_cover_pct,land_cover
0,patch_1.tif,24.517396,51.253355,749.956522,848.782609,Deserts & Xeric Shrublands,Qatar,Arabian-Persian Gulf coastal plain desert,22.0,0.445833,...,Palearctic,0.0,8.3,Jarayan Al Batnah,27.400000,41.500000,12.900000,28.600000,0.0,Bare/Sparse Vegetation
1,patch_2.tif,42.969783,112.248149,992.913043,1298.173913,Deserts & Xeric Shrublands,China,Eastern Gobi desert steppe,1097.0,0.437500,...,Palearctic,0.0,8.1,Nei Mongol Zizhiqu,3.900000,28.799999,-23.299999,52.099998,0.0,Bare/Sparse Vegetation
2,patch_3.tif,-17.116714,19.013767,2899.130435,4440.217391,"Tropical & Subtropical Grasslands, Savannas & ...",Angola,Zambezian Baikiaea woodlands,1157.0,0.249167,...,Afrotropic,5.0,7.5,Cuando Cubango,22.200001,34.000000,6.300000,27.700001,3.0,Shrubland
3,patch_5.tif,41.529312,23.845330,2439.173913,4137.739130,Temperate Broadleaf & Mixed Forests,Bulgaria,Balkan mixed forests,469.0,0.455833,...,Palearctic,20.0,7.5,Blagoevgrad,12.000000,28.400000,-1.900000,30.299999,7.0,Cropland
4,patch_7.tif,27.616751,26.800696,901.260870,1115.478261,Deserts & Xeric Shrublands,Egypt,East Sahara Desert,121.0,0.385833,...,Palearctic,0.0,8.2,New Valley,21.299999,37.500000,3.900000,33.599998,0.0,Bare/Sparse Vegetation
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69784,patch_99995.tif,-1.128649,125.110689,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69785,patch_99996.tif,80.721450,44.418605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69786,patch_99997.tif,-73.608960,37.202655,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69787,patch_99998.tif,31.619386,60.960150,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
output_dim = 256
satclip_embeddings = torch.zeros(len(satclip_df), output_dim).to(device)

with torch.no_grad():
    locations = satclip_df[["lon", "lat"]].values.astype(np.float64)
    locations = torch.tensor(locations, device=device) 
    
    dataloader = DataLoader(locations, batch_size=512, shuffle=False)
    i = 0
    for batch in tqdm(dataloader, total=len(dataloader), desc="Encoding location embeddings"):
        satclip_embeddings[i:i+batch.shape[0]] = model(batch)
        i += batch.shape[0]
        
        if i % 5120 == 0:  # every 10 batches with batch_size=512
            torch.cuda.empty_cache()

satclip_embeddings = torch.nn.functional.normalize(satclip_embeddings, dim=1) 

Encoding location embeddings:   0%|          | 0/137 [00:00<?, ?it/s]

Encoding location embeddings: 100%|██████████| 137/137 [00:02<00:00, 57.02it/s]


In [11]:
satclip_embeddings.shape

torch.Size([69789, 256])

In [12]:
satclip_embeddings

tensor([[-0.1220, -0.1057, -0.0182,  ..., -0.0259, -0.0530,  0.0024],
        [-0.1040,  0.0159,  0.0311,  ...,  0.0866,  0.0764,  0.0314],
        [ 0.0651, -0.0117,  0.1088,  ..., -0.0477, -0.0213,  0.0031],
        ...,
        [-0.0210,  0.1363,  0.0179,  ...,  0.0284, -0.0190, -0.0120],
        [-0.1209, -0.0184,  0.0039,  ..., -0.0504, -0.1115,  0.0035],
        [-0.1332, -0.0835,  0.0761,  ...,  0.0007,  0.0157, -0.0102]])

In [13]:
torch.save(satclip_embeddings.cpu(), "satclip_embeddings.pt")